# Kabyle-XLS-R Zero-Shot Evaluation on Tarifit

**Model:** `Akashpb13/Kabyle_xlsr`  
**Evaluation:** cleaned 128-example Tarifit validation subset  
**Decoding:** greedy CTC, no language model, no Kabyle→Tarifit orthographic mapping  
**Primary metric:** CER; WER is also reported.

In [ ]:
# Cell 1 — Mount Google Drive and check the runtime

from google.colab import drive
drive.mount("/content/drive")

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
        "GB"
    )
else:
    print("GPU unavailable. The notebook can run on CPU, but inference will be slower.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cpu
GPU unavailable. The notebook can run on CPU, but inference will be slower.


In [ ]:
# Cell 2 — Install the required packages

!pip -q install -U transformers accelerate datasets jiwer soundfile tqdm
!pip -q install "pandas==2.2.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 81.2 MB/s eta 0:00:00


In [ ]:
# Cell 3 — Define project paths and experiment configuration

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

VAL_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mms_corpus_v1_1"
    / "validation"
)

TARIFIT_VOCAB_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mms_tokenizer_v1_1"
    / "vocab.json"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "kabyle_xlsr_zero_shot"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

KABYLE_MODEL_ID = "Akashpb13/Kabyle_xlsr"

# These five validation pairs were verified as severely audio/text-misaligned.
BAD_VAL_INDICES = {111, 118, 126, 127, 130}

print("Validation directory exists:", VAL_DIR.exists())
print("Tarifit vocabulary exists:", TARIFIT_VOCAB_PATH.exists())
print("Results directory:", RESULTS_DIR)

Validation directory exists: True
Tarifit vocabulary exists: True
Results directory: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/kabyle_xlsr_zero_shot


In [ ]:
# Cell 4 — Load the fixed validation dataset and create the cleaned 128-example subset

from datasets import load_from_disk

val_ds = load_from_disk(str(VAL_DIR))

valid_indices = [
    i for i in range(len(val_ds))
    if i not in BAD_VAL_INDICES
]

val_clean_ds = val_ds.select(valid_indices)

print("Original validation examples:", len(val_ds))
print("Excluded corrupted examples:", sorted(BAD_VAL_INDICES))
print("Clean validation examples:", len(val_clean_ds))

assert len(val_ds) == 133
assert len(val_clean_ds) == 128

Original validation examples: 133
Excluded corrupted examples: [111, 118, 126, 127, 130]
Clean validation examples: 128


In [ ]:
# Cell 5 — Load the Tarifit reference tokenizer

from transformers import Wav2Vec2CTCTokenizer

tarifit_tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=str(TARIFIT_VOCAB_PATH),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

print("Tarifit tokenizer size:", len(tarifit_tokenizer))

reference_example = tarifit_tokenizer.decode(
    val_clean_ds[0]["labels"],
    group_tokens=False
)

print("First decoded reference:")
print(reference_example)

Tarifit tokenizer size: 38
First decoded reference:
ssalamuɛlikum necc meryem


In [ ]:
# Cell 6 — Load the Kabyle XLS-R processor and CTC model

from transformers import AutoProcessor, AutoModelForCTC

kabyle_processor = AutoProcessor.from_pretrained(KABYLE_MODEL_ID)

kabyle_model = AutoModelForCTC.from_pretrained(
    KABYLE_MODEL_ID
).to(device)

kabyle_model.eval()

print("Model:", KABYLE_MODEL_ID)
print("Parameters:", f"{kabyle_model.num_parameters():,}")
print("Kabyle output vocabulary size:", kabyle_model.config.vocab_size)
print(
    "Sampling rate:",
    getattr(kabyle_processor.feature_extractor, "sampling_rate", "unknown")
)

preprocessor_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.04k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/221 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.26GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Model: Akashpb13/Kabyle_xlsr
Parameters: 315,497,145
Kabyle output vocabulary size: 57
Sampling rate: 16000


In [ ]:
# Cell 7 — Decode all 128 Tarifit references

references = [
    tarifit_tokenizer.decode(
        example["labels"],
        group_tokens=False
    ).strip()
    for example in val_clean_ds
]

print("Decoded references:", len(references))

for i in range(3):
    print(f"{i}: {references[i]}")

Decoded references: 128
0: ssalamuɛlikum necc meryem
1: aqay ruxxa tnayn uɛecrin sana di hulanda
2: mercex ak nmis n jjiran usiɣ d zi lmeɣrib umi ira aqqay di lmeɣrib ira qqaɣas ad raḥaɣ a urupa ad ggex ad ggex maca


In [ ]:
# Cell 8 — Run Kabyle-XLS-R zero-shot inference with greedy CTC decoding

import torch
from tqdm.auto import tqdm

predictions = []

kabyle_model.eval()

for example in tqdm(val_clean_ds, desc="Kabyle-XLS-R zero-shot"):
    # The cached input_values are 16 kHz Wav2Vec2-style normalized waveform values.
    # We feed them directly to the CTC model and do not normalize them a second time.
    input_values = torch.tensor(
        example["input_values"],
        dtype=torch.float32
    ).unsqueeze(0).to(device)

    with torch.inference_mode():
        logits = kabyle_model(input_values=input_values).logits

    predicted_ids = torch.argmax(logits, dim=-1)

    prediction = kabyle_processor.batch_decode(
        predicted_ids
    )[0]

    predictions.append(prediction)

print("Predictions generated:", len(predictions))

for i in range(5):
    print("=" * 80)
    print("REFERENCE:", references[i])
    print("PREDICTION:", predictions[i])

Kabyle-XLS-R zero-shot:   0%|          | 0/128 [00:00<?, ?it/s]

Predictions generated: 128
REFERENCE: ssalamuɛlikum necc meryem
PREDICTION: assalam u ɛlikum necca mmeryam
REFERENCE: aqay ruxxa tnayn uɛecrin sana di hulanda
PREDICTION: a aqlya ruxxan tnayen n uɛecrin sanadiholanda
REFERENCE: mercex ak nmis n jjiran usiɣ d zi lmeɣrib umi ira aqqay di lmeɣrib ira qqaɣas ad raḥaɣ a urupa ad ggex ad ggex maca
PREDICTION: a mercex aggamis n jiran usixezil meɣrip umiraqaydi lmeɣribh ilaq-aɣ-c adraḥ a ilaqaɣ-as ad raḥeɣ uruba deg ex deg ax maca
REFERENCE: umi wsiɣd dda ufix manayenni wa ǧi ca min ira ɣari ddhi rɛqel inu
PREDICTION: umi usiɣ dda ufixh ufixman ayen i waǧic-a mi rirɣridi di leɛqel-inu
REFERENCE: a nec mammec ira ǧjix di lmeɣrib waǧi manayenni uffix dda
PREDICTION: a nnecca ma amma amciraǧix di lmeɣri-m twaǧi mi anayen iufixeddaɣ


In [ ]:
# Cell 9 — Apply the common minimal evaluation normalization

import re
import unicodedata

def eval_normalize(text):
    text = unicodedata.normalize("NFC", str(text))
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

references_eval = [
    eval_normalize(text)
    for text in references
]

predictions_eval = [
    eval_normalize(text)
    for text in predictions
]

print("Normalization completed.")

Normalization completed.


In [ ]:
# Cell 10 — Compute corpus-level WER and CER

from jiwer import wer, cer

kabyle_xlsr_wer = wer(
    references_eval,
    predictions_eval
)

kabyle_xlsr_cer = cer(
    references_eval,
    predictions_eval
)

print("=" * 65)
print("KABYLE-XLS-R ZERO-SHOT — CLEAN TARIFIT VALIDATION")
print("=" * 65)
print("Segments :", len(references_eval))
print(f"WER      : {kabyle_xlsr_wer * 100:.2f}%")
print(f"CER      : {kabyle_xlsr_cer * 100:.2f}%")

KABYLE-XLS-R ZERO-SHOT — CLEAN TARIFIT VALIDATION
Segments : 128
WER      : 100.33%
CER      : 54.87%


In [ ]:
# Cell 11 — Compute per-segment errors and save the zero-shot results

import pandas as pd
from jiwer import wer, cer

segment_wer = [
    wer(ref, pred)
    for ref, pred in zip(references_eval, predictions_eval)
]

segment_cer = [
    cer(ref, pred)
    for ref, pred in zip(references_eval, predictions_eval)
]

results_df = pd.DataFrame({
    "clean_validation_index": range(len(val_clean_ds)),
    "original_validation_index": valid_indices,
    "duration_seconds": [
        example["input_length"] / 16000
        for example in val_clean_ds
    ],
    "reference": references,
    "prediction_raw": predictions,
    "reference_eval": references_eval,
    "prediction_eval": predictions_eval,
    "wer": segment_wer,
    "cer": segment_cer,
})

prediction_file = RESULTS_DIR / "kabyle_xlsr_zero_shot_validation_128.csv"
results_df.to_csv(prediction_file, index=False, encoding="utf-8")

summary_df = pd.DataFrame([{
    "experiment": "Kabyle-XLS-R zero-shot -> Tarifit",
    "model": KABYLE_MODEL_ID,
    "split": "clean_validation_128",
    "segments": len(val_clean_ds),
    "decoding": "greedy CTC",
    "language_model": False,
    "orthographic_mapping": False,
    "wer_percent": kabyle_xlsr_wer * 100,
    "cer_percent": kabyle_xlsr_cer * 100,
}])

summary_file = RESULTS_DIR / "kabyle_xlsr_zero_shot_summary.csv"
summary_df.to_csv(summary_file, index=False)

print("Saved predictions:", prediction_file)
print("Saved summary:", summary_file)
display(summary_df)

Saved predictions: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/kabyle_xlsr_zero_shot/kabyle_xlsr_zero_shot_validation_128.csv
Saved summary: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/kabyle_xlsr_zero_shot/kabyle_xlsr_zero_shot_summary.csv


,experiment,model,split,segments,decoding,language_model,orthographic_mapping,wer_percent,cer_percent
0,Kabyle-XLS-R zero-shot -> Tarifit,Akashpb13/Kabyle_xlsr,clean_validation_128,128,greedy CTC,False,False,100.33389,54.866327


In [ ]:
# Cell 12 — Inspect the best and worst zero-shot examples by CER

display(
    results_df.sort_values("cer")
    [["original_validation_index", "duration_seconds", "reference", "prediction_raw", "cer"]]
    .head(10)
)

display(
    results_df.sort_values("cer", ascending=False)
    [["original_validation_index", "duration_seconds", "reference", "prediction_raw", "cer"]]
    .head(10)
)

,original_validation_index,duration_seconds,reference,prediction_raw,cer
57,57,0.944,qlil,qlil,0.000000
50,50,3.352,ɛad qa rux rux mecḥar n ssna,ɛad qeṛrux ṛux mecḥal n ssne,0.178571
33,33,11.696,uca melmi tusa lweqt iɛni thegid di lbal nnem ...,uca melmi t-tusa lweqt yɛni teggid-d i lbalen ...,0.218045
14,14,12.624,ṣafi igga manayenni i lwalidin nnes waǧi netta...,safi igga manayenni yelwalidin-nnes waǧir n d ...,0.224044
56,56,6.608,yeɛni mani tuɣ tellid ked di taddat nni ak d e...,yeɛni man-ittuɣ tedlidakkt deg taddart-nni akk...,0.235294
0,0,3.280,ssalamuɛlikum necc meryem,assalam u ɛlikum necca mmeryam,0.240000
10,10,11.472,usiɣd umi usiɣd necc usiɣd ddirikt ɣar taddart...,usiɣ-d umi usiɣ-d nec usiɣ ddirikt ɣa ttaddatn...,0.244604
49,49,14.808,netta wa cem ixes ca wah iwa qa manayenni qaɣa...,neta wa kem yixeccala iwaqaman ayen iqqaɣ-as ṛ...,0.245614
17,17,10.664,axmi waǧi taddat inu axmi wa lmuhim yemmas ira...,axem-iwaǧi ttaddat-inu axemiwa lmuhim yemma-s ...,0.247706
8,8,1.968,lmuhim wsiɣd,muhimwsiɣ-d,0.250000


,original_validation_index,duration_seconds,reference,prediction_raw,cer
123,125,13.0,qqarn as subḥanllah,anellan la yennaɣatwatmunek tɣatwatmun ak akk ...,10.105263
101,101,11.0,yiwey itent s trata,muhim ara meyehnti-au elaxirih yenn-as i tenni...,7.473684
124,128,20.0,ittsemma babas d yemmas ḥaqiqiyyin,n wahi m ruḥeɣ-as babas-nni yeteres dg fellah...,7.117647
116,117,26.0,tteṭṭes di deg winat tteṭṭes di mamek d as qqa...,tres tɛicic n mḥayat cwayet teqsaḥ muhim degg ...,6.050847
126,131,9.0,ɣanim d aziza ad t qessen,sen aahan ssekkensen ǧ-nsen ettmaḥkem xassen-t...,5.600000
114,115,9.0,ggin t as deg uqemmum i temɣart nni yurwen,almiesreggin deg yinesut yidammen deg uqmumumu...,2.380952
109,109,4.0,mya di mya ad xasent yazzer,er aramit-nni tkemmer awar-nnes tezzed aḥram ɣ...,1.592593
110,110,9.0,arami tenni tkemmer awar nnes tejja dd aḥram ɣ...,nyatur akin tinḍen i ɛejzen-t ig nfekkant nyen...,1.569231
105,105,10.0,ten nnezḍni i d as innan ad as dd jjeɣ aḥenjir...,at atelaḍ rjun arɣa al ɣ ateɛrul yenu asssinn ...,1.462687
89,89,5.0,beddent ṭṭarf i ufeddan nni n yirden,faten-as yectennas ef dda temma bayen ilbab-nn...,1.250000


## Experiment conclusion

The Kabyle-XLS-R model was evaluated directly on the cleaned Tarifit validation set without any Tarifit fine-tuning, language model, or orthographic post-processing. The evaluation was performed on 128 validation segments using greedy CTC decoding.

The model obtained:

- **WER: 100.33%**
- **CER: 54.87%**

The very high WER shows that the model rarely produces fully correct Tarifit word sequences. However, the substantially lower CER indicates that its predictions still preserve a meaningful amount of character-level similarity to the Tarifit references.

This suggests that the Kabyle-trained XLS-R model transfers some useful acoustic and graphemic knowledge to Tarifit despite the language and orthographic differences between the two varieties. The result therefore provides evidence of partial cross-lingual transfer at character level, while also showing that zero-shot Kabyle ASR is not sufficient for accurate Tarifit transcription.